## Ingestion Pipeline

In [1]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader


C:\Users\Sai Kiran Sugurthi\AppData\Local\Temp\ipykernel_18936\2748893930.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader
C:\Users\Sai Kiran Sugurthi\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_all_pdfs():
    folder_path="data/pdfs"
    num_docs=0
    all_docs=[]

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            #create complete file path
            pdf_path=os.path.join(folder_path,filename)

            loader=PyPDFLoader(pdf_path)
            doc=loader.load()

            all_docs.extend(doc)
            num_docs+=1
    print("total pdfs:",num_docs)
    print("total pages:",len(all_docs))
    return all_docs
        

In [3]:
all_pdf_docs=load_all_pdfs()

total pdfs: 1
total pages: 21


In [4]:
#CREATING CHUNKS FROM DOCUMENTS

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_docs(documents,chunk_size=500,chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
    chunks=text_splitter.split_documents(documents)
    return chunks;
    
    

In [6]:
chunks=split_docs(all_pdf_docs)
# chunks

In [7]:
len(chunks)

244

In [8]:
!pip install sentence-transformers


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## CREATING EMBEDDINGS

In [9]:
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("loading model ...",self.model_name)
        self.model=SentenceTransformer(self.model_name)
        print("embedding Dimensions : ",self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self,text):
        embeddings=self.model.encode(text,show_progress_bar=True)
        print("Embeddings shape :",embeddings.shape)
        return embeddings
        


In [10]:
embedding_manager=EmbeddingManager()

loading model ... all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3784.51it/s]


embedding Dimensions :  384


C:\Users\Sai Kiran Sugurthi\AppData\Local\Temp\ipykernel_18936\3851264641.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding Dimensions : ",self.model.get_sentence_embedding_dimension())


## Vector Store


In [11]:
import chromadb
import uuid

In [12]:
class VectorStoreManager:
    def __init__(self,persistent_directory="data/vector_store",collection_name="pdf_documents"):
        self.collection_name=collection_name;
        self.persistent_directory=persistent_directory
        self.collection=None
        self.client=None

        self._initialise_store()

    def _initialise_store(self):
        os.makedirs(self.persistent_directory,exist_ok=True)

        # Create a Client
        self.client=chromadb.PersistentClient(path=self.persistent_directory)

        #Create the collection
        self.collection=self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description":"vector store collection for pdf embeddings in RAG"}
        )

        print("initialised the vector store with collection :",self.collection_name)

    def add_documents(self,documents,embeddings):
        if len(documents)!=len(embeddings):
            raise ValueError("Docs length doesn't match embeddings length")

        ids=[]
        all_metadata=[]
        documents_content=[]
        embeddings_list=[]
        
        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata["doc_index"]=i
            metadata["content_length"]=len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embedding)

        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list)

        print("total documets added in the vector store :",len(documents_content))
        print("docs in collection :",self.collection.count())
    

In [13]:
vectorstoremanager=VectorStoreManager()

initialised the vector store with collection : pdf_documents


In [70]:
#doc=> chunk => embeddings => vector store

In [14]:
texts=[doc.page_content for doc in chunks]

embeddings=embedding_manager.generate_embeddings(texts)
vectorstoremanager.add_documents(chunks,embeddings)

Batches: 100%|██████████| 8/8 [00:12<00:00,  1.54s/it]


Embeddings shape : (244, 384)
total documets added in the vector store : 244
docs in collection : 244
